In [3]:
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import GaussianNB
import matplotlib.pyplot as plt
import re
import pickle

In [4]:
df = pd.read_csv("./Restaurant_Reviews.tsv", sep="\t")

In [5]:
df.head()

,Review,Liked
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1


In [6]:
df.isnull().sum()

Review    0
Liked     0
dtype: int64

In [7]:
df['Liked'].value_counts()

Liked
1    500
0    500
Name: count, dtype: int64

In [8]:
df.shape

(1000, 2)

In [9]:
corpus = []
for i in range(1000):
        r = re.sub('[^a-zA-Z]', ' ', df['Review'][i]).lower().split()
        ps = PorterStemmer()
        sw = stopwords.words('english')
        if 'not' in sw:
            sw.remove('not')
        r = [ps.stem(w) for w in r if w not in sw]
        corpus.append(' '.join(r))

In [22]:
corpus[1:10]

['crust not good',
 'not tasti textur nasti',
 'stop late may bank holiday rick steve recommend love',
 'select menu great price',
 'get angri want damn pho',
 'honeslti tast fresh',
 'potato like rubber could tell made ahead time kept warmer',
 'fri great',
 'great touch']

In [11]:
cv = CountVectorizer(max_features=1500)

In [12]:
cv

CountVectorizer(max_features=1500)

In [13]:
X = cv.fit_transform(corpus).toarray()
y = df.iloc[:, -1].values

In [14]:
X[1:5]

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [15]:
y[1:5]

array([0, 0, 1, 1])

In [16]:
model = GaussianNB()

In [17]:
model.fit(X, y)


GaussianNB()

In [18]:
def predict(text):
    r = re.sub('[^a-zA-Z]', ' ', text).lower().split()
    ps = PorterStemmer()
    sw = stopwords.words('english')
    if 'not' in sw:
        sw.remove('not')

    r = [ps.stem(w) for w in r if w not in sw]
    r = ' '.join(r)

    X = cv.transform([r]).toarray()
    res = model.predict(X)[0]

    if "not" in r:
        res = abs(res - 1)

    return res

In [19]:
predict("good")

np.int64(1)

In [20]:
predict("bad")

np.int64(0)

In [21]:
predict("servic prompt")

np.int64(1)

In [23]:
with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)
  
  
with open('CountVectorizer.pkl', 'wb') as f:
    pickle.dump(cv, f)  